In [7]:
# ==============================
# 📌 Project: Customer Shopping Behavior Analysis
# ==============================

# Importing required libraries
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


In [8]:
# Loading dataset
file_path = r"C:\Users\Meet Joshi\Desktop\Projects\main projects\Project 2 customer shopping behaviour\Raw Files\customer_shopping_behavior.csv"

df = pd.read_csv(file_path)

print("✅ Dataset loaded successfully")

✅ Dataset loaded successfully


In [9]:
# ==============================
# 🔍 Initial Data Overview
# ==============================

print("\n📌 Dataset Shape:")
print(df.shape)

print("\n📌 Data Info:")
print(df.info())

print("\n📌 Missing Values:")
print(df.isnull().sum())

print("\n📌 Sample Data:")
print(df.head())


📌 Dataset Shape:
(3900, 18)

📌 Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Co

In [10]:
# ==============================
# 🧹 Column Cleaning
# ==============================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', '_', regex=True)
)

print("✅ Column names standardized")
print(df.columns)

✅ Column names standardized
Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_usd', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')


In [11]:
# Renaming columns for clarity
df = df.rename(columns={
    'purchase_amount_usd': 'purchase_amount'
})

print("✅ Columns renamed successfully")

✅ Columns renamed successfully


In [12]:
print("\n📌 Updated Columns:")
print(df.columns)

print("\n📌 Data Preview After Cleaning:")
print(df.head())


📌 Updated Columns:
Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

📌 Data Preview After Cleaning:
   customer_id  age gender item_purchased  category  purchase_amount  \
0            1   55   Male         Blouse  Clothing               53   
1            2   19   Male        Sweater  Clothing               64   
2            3   50   Male          Jeans  Clothing               73   
3            4   21   Male        Sandals  Footwear               90   
4            5   45   Male         Blouse  Clothing               49   

        location size      color  season  review_rating subscription_status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    

In [13]:
# ==============================
# 🔍 Missing Values Before Cleaning
# ==============================

missing_before = df.isnull().sum()

print("\n📌 Missing Values Before:")
print(missing_before[missing_before > 0])


📌 Missing Values Before:
review_rating    37
dtype: int64


In [14]:
# ==============================
# ⭐ Handling Missing Values: Review Rating
# ==============================

df['review_rating'] = df.groupby('category')['review_rating']\
    .transform(lambda x: x.fillna(x.median()))

In [15]:
# Filling any remaining nulls with overall median
df['review_rating'] = df['review_rating'].fillna(df['review_rating'].median())

In [16]:
# ==============================
# ✅ Validation After Missing Value Handling
# ==============================

missing_after = df.isnull().sum()

print("\n📌 Missing Values After:")
print(missing_after[missing_after > 0])


📌 Missing Values After:
Series([], dtype: int64)


In [17]:
print("\n📊 Review Rating Summary:")
print(df['review_rating'].describe())


📊 Review Rating Summary:
count    3900.000000
mean        3.750051
std         0.713590
min         2.500000
25%         3.100000
50%         3.800000
75%         4.400000
max         5.000000
Name: review_rating, dtype: float64


In [18]:
# ==============================
# 👥 Age Group Segmentation
# ==============================

labels = ['young_adult', 'adult', 'middle_aged', 'senior']

df['age_group'] = pd.qcut(
    df['age'],
    q=4,
    labels=labels,
    duplicates='drop'
)

print("✅ Age group created successfully")

✅ Age group created successfully


In [19]:
# ==============================
# 🔁 Purchase Frequency Conversion
# ==============================

frequency_mapping = {
    'weekly': 7,
    'bi-weekly': 14,
    'fortnightly': 14,
    'monthly': 30,
    'every 3 months': 90,
    'quarterly': 90,
    'annually': 365
}

df['purchase_frequency_days'] = (
    df['frequency_of_purchases']
    .str.strip()
    .str.lower()
    .map(frequency_mapping)
)

print("✅ Purchase frequency converted to days")

✅ Purchase frequency converted to days


In [20]:
# ==============================
# 📊 Customer Activity Metric
# ==============================

df['purchase_per_month'] = 30 / df['purchase_frequency_days']

In [21]:
# ==============================
# 💰 Customer Value Segmentation
# ==============================

df['customer_value'] = pd.qcut(
    df['purchase_amount'],
    q=3,
    labels=['low', 'medium', 'high']
)

In [22]:
# Spending category
df['spending_category'] = pd.cut(
    df['purchase_amount'],
    bins=[0, 50, 150, 500],
    labels=['low_spender', 'medium_spender', 'high_spender']
)

In [23]:
print("\n📌 Feature Check:")
print(df[['age_group', 'purchase_frequency_days', 'purchase_per_month', 'customer_value']].head())


📌 Feature Check:
     age_group  purchase_frequency_days  purchase_per_month customer_value
0  middle_aged                       14            2.142857         medium
1  young_adult                       14            2.142857         medium
2  middle_aged                        7            4.285714         medium
3  young_adult                        7            4.285714           high
4  middle_aged                      365            0.082192         medium


In [24]:
# ==============================
# 🧹 Dropping Unnecessary Columns
# ==============================

cols_to_drop = ['promo_code_used']

df = df.drop(columns=cols_to_drop, errors='ignore')

print("✅ Unnecessary columns dropped")

✅ Unnecessary columns dropped


In [25]:
# ==============================
# ✅ Final Dataset Validation
# ==============================

print("\n📌 Final Shape:")
print(df.shape)

print("\n📌 Final Info:")
print(df.info())

print("\n📌 Final Missing Values:")
print(df.isnull().sum())

print("\n📌 Final Preview:")
print(df.head())


📌 Final Shape:
(3900, 22)

📌 Final Info:
<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   customer_id              3900 non-null   int64   
 1   age                      3900 non-null   int64   
 2   gender                   3900 non-null   str     
 3   item_purchased           3900 non-null   str     
 4   category                 3900 non-null   str     
 5   purchase_amount          3900 non-null   int64   
 6   location                 3900 non-null   str     
 7   size                     3900 non-null   str     
 8   color                    3900 non-null   str     
 9   season                   3900 non-null   str     
 10  review_rating            3900 non-null   float64 
 11  subscription_status      3900 non-null   str     
 12  shipping_type            3900 non-null   str     
 13  discount_applied         3900 no

In [26]:
from sqlalchemy import create_engine

server = 'MJ9797\\MEET9797'
database = 'Customer_Shopping_Behaviour_Project'

engine = create_engine(
    f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

print("✅ Connected successfully")

✅ Connected successfully


In [27]:
conn = engine.connect()
print("✅ Connection test successful")
conn.close()

✅ Connection test successful


In [30]:
# Add age_sort column
df['age_sort'] = df['age_group'].map({
    'young_adult': 1,
    'adult': 2,
    'middle_aged': 3,
    'senior': 4
})

print("✅ age_sort added")

✅ age_sort added


In [31]:
df.to_sql(
    name='customer_data',
    con=engine,
    if_exists='replace',   
    index=False
)

print("✅ Data uploaded ")

✅ Data uploaded 
